# Olist E-commerce: Export Summary Tables for Tableau

Small, pre-aggregated tables for the dashboard. They contain **counts and sums only**
(not percentages or averages), so Tableau can calculate correct ratios at any level.

In [1]:
import duckdb

%load_ext sql
con = duckdb.connect("../data/olist.duckdb", read_only=True)
%sql con
%config SqlMagic.displaylimit = 50

## Order-level base table

One row per delivered order (Jan 2017 – Aug 2018) with revenue, delivery flags and the latest review.
Revenue is aggregated per order **before** joining, so there is no fan-out.
Saved as a JupySQL snippet (`order_level`) and reused by the exports below.

In [2]:
%%sql --save order_level --no-execute
WITH order_revenue AS (
    SELECT order_id,
           SUM(price) AS revenue
    FROM order_items
    GROUP BY order_id
),
latest_review AS (
    SELECT order_id, review_score
    FROM (
        SELECT order_id,
               review_score,
               ROW_NUMBER() OVER (PARTITION BY order_id
                                  ORDER BY review_answer_timestamp DESC, review_id) AS rn
        FROM reviews
    )
    WHERE rn = 1
)
SELECT o.order_id,
       date_trunc('month', o.order_purchase_timestamp)::DATE AS month,
       c.customer_state AS state,
       r.revenue,
       CASE WHEN o.order_delivered_customer_date IS NOT NULL THEN 1 ELSE 0 END AS has_delivery_date,
       CASE WHEN date_diff('day', o.order_estimated_delivery_date,
                                  o.order_delivered_customer_date) > 0 THEN 1 ELSE 0 END AS is_late,
       date_diff('day', o.order_estimated_delivery_date, o.order_delivered_customer_date) AS days_late,
       date_diff('day', o.order_purchase_timestamp, o.order_delivered_customer_date) AS delivery_days,
       lr.review_score
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_revenue r ON o.order_id = r.order_id
LEFT JOIN latest_review lr ON o.order_id = lr.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'

Running query in 'DuckDBPyConnection'

Skipping execution...

## monthly.csv: one row per month

In [3]:
%%sql monthly <<
SELECT month,
       COUNT(*) AS orders,
       ROUND(SUM(revenue), 2) AS revenue,
       SUM(has_delivery_date)::INTEGER AS orders_with_delivery_date,
       SUM(is_late)::INTEGER AS late_orders,
       COUNT(review_score) AS reviews,
       SUM(review_score)::INTEGER AS review_points
FROM order_level
GROUP BY month
ORDER BY month

Generating CTE with stored snippets: 'order_level'

Running query in 'DuckDBPyConnection'

In [4]:
monthly.DataFrame().to_csv("../dashboard/data/monthly.csv", index=False)
monthly

month,orders,revenue,orders_with_delivery_date,late_orders,reviews,review_points
2017-01-01,750,111798.36,750,22,741,3111
2017-02-01,1653,234223.4,1653,49,1643,6906


## states.csv: one row per state

Full state names plus a `country` column help Tableau place states on the map correctly.
The names are a separate snippet because JupySQL can't expand a snippet inside a query that has its own `WITH`.

In [5]:
%%sql --save state_names --no-execute
SELECT *
FROM (VALUES ('AC','Acre'), ('AL','Alagoas'), ('AP','Amapá'), ('AM','Amazonas'), ('BA','Bahia'),
             ('CE','Ceará'), ('DF','Distrito Federal'), ('ES','Espírito Santo'), ('GO','Goiás'),
             ('MA','Maranhão'), ('MT','Mato Grosso'), ('MS','Mato Grosso do Sul'), ('MG','Minas Gerais'),
             ('PA','Pará'), ('PB','Paraíba'), ('PR','Paraná'), ('PE','Pernambuco'), ('PI','Piauí'),
             ('RJ','Rio de Janeiro'), ('RN','Rio Grande do Norte'), ('RS','Rio Grande do Sul'),
             ('RO','Rondônia'), ('RR','Roraima'), ('SC','Santa Catarina'), ('SP','São Paulo'),
             ('SE','Sergipe'), ('TO','Tocantins')) AS t(state, state_name)

Running query in 'DuckDBPyConnection'

Skipping execution...

In [6]:
%%sql states <<
SELECT ol.state,
       n.state_name,
       'Brazil' AS country,
       COUNT(*) AS orders,
       ROUND(SUM(ol.revenue), 2) AS revenue,
       SUM(ol.has_delivery_date)::INTEGER AS orders_with_delivery_date,
       SUM(ol.is_late)::INTEGER AS late_orders,
       SUM(ol.delivery_days)::INTEGER AS total_delivery_days,
       COUNT(ol.review_score) AS reviews,
       SUM(ol.review_score)::INTEGER AS review_points
FROM order_level ol
JOIN state_names n ON ol.state = n.state
GROUP BY ol.state, n.state_name
ORDER BY revenue DESC

Generating CTE with stored snippets: 'order_level', and 'state_names'

Running query in 'DuckDBPyConnection'

In [7]:
states.DataFrame().to_csv("../dashboard/data/states.csv", index=False)
states

state,state_name,country,orders,revenue,orders_with_delivery_date,late_orders,total_delivery_days,reviews,review_points
SP,São Paulo,Brazil,40406,5055587.13,40399,1817,350527,40178,170619
RJ,Rio de Janeiro,Brazil,12310,1751433.85,12310,1495,187409,12172,48268


## categories.csv: one row per product category

Item-level grain (one order can contain several categories), so this query doesn't use `order_level`.

In [8]:
%%sql categories <<
SELECT replace(COALESCE(t.product_category_name_english, p.product_category_name, 'unknown'), '_', ' ') AS category,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(i.price), 2) AS revenue
FROM orders o
JOIN order_items i ON o.order_id = i.order_id
LEFT JOIN products p ON i.product_id = p.product_id
LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'
GROUP BY category
ORDER BY revenue DESC

Running query in 'DuckDBPyConnection'

In [9]:
categories.DataFrame().to_csv("../dashboard/data/categories.csv", index=False)
categories

category,orders,revenue
health beauty,8610,1229557.5
watches gifts,5491,1163465.91


## delivery_groups.csv: one row per delivery delay group

In [10]:
%%sql delivery_groups <<
SELECT CASE WHEN days_late <= -10 THEN '1. 10+ days early'
            WHEN days_late <= 0   THEN '2. 0-9 days early'
            WHEN days_late <= 3   THEN '3. 1-3 days late'
            WHEN days_late <= 7   THEN '4. 4-7 days late'
            WHEN days_late <= 14  THEN '5. 8-14 days late'
            ELSE '6. 15+ days late'
       END AS delivery_group,
       COUNT(*) AS reviews,
       SUM(review_score)::INTEGER AS review_points,
       SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END)::INTEGER AS bad_reviews
FROM order_level
WHERE has_delivery_date = 1
  AND review_score IS NOT NULL
GROUP BY delivery_group
ORDER BY delivery_group

Generating CTE with stored snippets: 'order_level'

Running query in 'DuckDBPyConnection'

In [11]:
delivery_groups.DataFrame().to_csv("../dashboard/data/delivery_groups.csv", index=False)
delivery_groups

delivery_group,reviews,review_points,bad_reviews
1. 10+ days early,61270,264965,5430
2. 0-9 days early,27912,117729,2817


## segments.csv: one row per RFM segment

Same logic as notebook 04 (with the `customer_unique_id` tie-breaker), exporting sums.

In [12]:
%%sql segments <<
WITH customer_orders AS (
    SELECT c.customer_unique_id,
           o.order_id,
           o.order_purchase_timestamp AS order_date,
           SUM(i.price) AS order_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp
),
rfm AS (
    SELECT customer_unique_id,
           date_diff('day', MAX(order_date), DATE '2018-09-01') AS recency_days,
           COUNT(*) AS frequency,
           SUM(order_value) AS monetary
    FROM customer_orders
    GROUP BY customer_unique_id
),
scored AS (
    SELECT *,
           NTILE(5) OVER (ORDER BY recency_days DESC, customer_unique_id) AS r_score,
           NTILE(5) OVER (ORDER BY monetary, customer_unique_id) AS m_score
    FROM rfm
),
segmented AS (
    SELECT *,
           CASE WHEN frequency >= 2                 THEN 'Repeat customers'
                WHEN r_score >= 4 AND m_score >= 4  THEN 'Recent big spenders'
                WHEN r_score <= 2 AND m_score >= 4  THEN 'Lapsed big spenders'
                WHEN r_score <= 2 AND m_score <= 2  THEN 'Lost low spenders'
                ELSE 'Regular'
           END AS segment
    FROM scored
)
SELECT segment,
       COUNT(*) AS customers,
       ROUND(SUM(monetary), 2) AS revenue,
       SUM(recency_days)::INTEGER AS total_recency_days
FROM segmented
GROUP BY segment
ORDER BY revenue DESC

Running query in 'DuckDBPyConnection'

In [13]:
segments.DataFrame().to_csv("../dashboard/data/segments.csv", index=False)
segments

segment,customers,revenue,total_recency_days
Regular,46931,4191182.16,8857567
Recent big spenders,14441,3856546.44,1356511


## Check: do the files agree?

In [14]:
import pandas as pd

m = pd.read_csv("../dashboard/data/monthly.csv")
s = pd.read_csv("../dashboard/data/states.csv")
c = pd.read_csv("../dashboard/data/categories.csv")
g = pd.read_csv("../dashboard/data/segments.csv")

print("Revenue  monthly:", round(m.revenue.sum()), "| states:", round(s.revenue.sum()), "| categories:", round(c.revenue.sum()))
print("Orders   monthly:", m.orders.sum(), "| states:", s.orders.sum())
print("Late %:", round(100 * m.late_orders.sum() / m.orders_with_delivery_date.sum(), 1))
print("Avg review:", round(m.review_points.sum() / m.reviews.sum(), 2))
print("Customers:", g.customers.sum(),
      "| repeat %:", round(100 * g.loc[g.segment == "Repeat customers", "customers"].sum() / g.customers.sum(), 1))

Revenue  monthly: 13181027 | states: 13181027 | categories: 13181027
Orders   monthly: 96211 | states: 96211
Late %: 6.8
Avg review: 4.16
Customers: 93104 | repeat %: 3.0


In [15]:
con.close()

## Exported files (dashboard/data/)

| File | Grain (one row =) | Rows | Used for |
|---|---|---|---|
| monthly.csv | month | 20 | KPIs, revenue trend, late % and reviews over time |
| states.csv | state | 27 | maps: revenue and late % by state |
| categories.csv | product category | 74 | top categories bar chart |
| delivery_groups.csv | delivery delay group | 6 | review score vs delay |
| segments.csv | RFM segment | 5 | customer segments |

**Design rule:** only counts and sums are exported. Percentages and averages are calculated in
Tableau as SUM(numerator) / SUM(denominator), which avoids the "average of averages" mistake.

**Check:** revenue (R$ 13,181,027) and orders (96,211) match across all files and earlier steps.

## SQL lessons
- Aggregate before joining (`order_revenue` has one row per order) to avoid fan-out.
- JupySQL snippets only work inside `SELECT` queries and can't be expanded inside a query with its own `WITH`,
  so results are stored with `<<` and saved with pandas.
- `::INTEGER` turns sums like `750.0` into whole numbers.